In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=512,
    chunk_overlap=128,
    length_function=len,
    is_separator_regex=False,
)

In [3]:
from openai import OpenAI
import os 
from dotenv import load_dotenv
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [4]:
###INSERT CONTEXT
def insertContext(text, full_doc):
    content = text.page_content
    res  = client.chat.completions.create(
    model="gpt-4o-mini", ###change to 4o
    store=True,
    messages=[
        {"role": "system", "content": f"please generate appropriate context for the provided chunk. Please note that the added context should include information that is in the following document but not in the chunk. Documnet: \n {full_doc}."},
        {"role": "user", "content": f"chunk: {content}"}
    ])
    context = res.choices[0].message.content
    cached_tokens = res.usage.prompt_tokens_details.cached_tokens
    return content + context 

In [5]:
from transformers import BertTokenizer

# load bert tokenizer from huggingface
tokenizer = BertTokenizer.from_pretrained(
    'bert-base-uncased'
)

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [6]:
from collections import Counter

def build_dict(input_batch):
  # store a batch of sparse embeddings
    sparse_emb = []
    # iterate through input batch
    for token_ids in input_batch:
        # convert the input_ids list to a dictionary of key to frequency values
        d = dict(Counter(token_ids))
        tokenids = list(set(token_ids))
        # remove special tokens and append sparse vectors to sparse_emb list
        # sparse_emb.append({key: d[key] for key in d if key not in [101, 102, 103, 0]})
        sparse_emb.append({"indices":tokenids, "values":[float(d[id]) for id in tokenids]})
    # return sparse_emb list
    return sparse_emb

In [7]:
def generate_sparse_vectors(context_batch):
    input_ids = tokenizer(
    context_batch, padding=True, truncation=True,
     max_length=512
)["input_ids"]
    sparse_embeds = build_dict(input_ids)
    return sparse_embeds

In [8]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
pc = Pinecone(api_key= os.getenv("PINECONE_API_KEY_500"))
index_name = "contextual-retriever"


In [ ]:
indices = []
for index in pc.list_indexes():
    indices.append(index["name"])

if index_name in indices :
    print(f"index {index_name} already exists!")
    index = pc.Index("contextual-retriever")
else:
    pc.create_index(
  name=index_name,
  dimension=3072,
  metric="dotproduct",
  spec=ServerlessSpec(
    cloud="aws",
    region="us-east-1"
  ),
  deletion_protection="disabled"
)
    print(f"index {index_name} created")



In [9]:
def generate_dense_embeddings(texts):
    content = [text.page_content for text in texts]
    response =  client.embeddings.create(
    input=content,
    model="text-embedding-3-large")
    return [item.embedding for item in response.data]



In [10]:
import json 

with open(r"../../data/url_content_mapping.json", "r",encoding="utf-8") as file:
    data = json.load(file)
context =""
texts = []
for content in data:
    context += f"Service URL: {content["url"]} \r\n Content: {content["content"]} \r\n"
    texts.extend(text_splitter.create_documents([content["content"]], [{"source":content["url"]}]))



In [11]:
url_dict={}
for item in data:
    url_dict[item["url"]]= item["content"]



In [ ]:
batch_requests= []
for i,text in enumerate(texts[0:3]):
    system_prompt = f"please generate appropriate context for the provided chunk. Please note that the added context should include general information about the service, information about mohap and information that is missing from the document. For example if the chunk is talking about the fees and does not mention the service name add the service name and the general purpose. If requirements are mentioned without names and restrictions mention them. Documnet: \n {url_dict[text.metadata["source"]]}."
    batch_requests.append({"custom_id":f"inser_context_{i}", "method":"POST", "url": "/v1/chat/completions", "body": {"model": "gpt-4o", "messages": [{"role": "system", "content": system_prompt},{"role": "user", "content":  f"chunk: {text.page_content}"}],"max_tokens": 250}})

In [ ]:
import json

with open('Batch_Requests_0-3.jsonl', 'w', encoding='utf-8') as f:
    for item in batch_requests:
        json_line = json.dumps(item)
        f.write(json_line + '\n')

In [ ]:
contexts = []

for text in texts:
    contexts.append(text.page_content)

In [ ]:
ids = [str(x) for x in range(len(texts))]
# add context passages as metadata
meta = [{'context': text.page_content} for text in texts]
# create dense vectors
dense_embeds = generate_dense_embeddings(texts)
print(f"length of dense: {len(dense_embeds)}")
# create sparse vectors
sparse_embeds = generate_sparse_vectors(contexts)
print(f"length of sparse: {len(sparse_embeds)}")
vectors = []
# loop through the data and create dictionaries for uploading documents to pinecone index
for _id, sparse, dense, metadata in zip(ids, sparse_embeds, dense_embeds, meta):
    vectors.append({
        'id': _id,
        'sparse_values': sparse,
        'values': dense,
        'metadata': metadata
    })

In [ ]:
print("about to updsert value")
index.upsert(vectors=vectors)
print("upserted")